# Konfigurovateľný pipeline pre koronálne diery

Tento notebook demonštruje štandardný pipeline praktickej časti práce. Notebook je riadený konfiguračným súborom vo formáte JSON.

Na začiatku notebooku sa zvolí jeden konfiguračný súbor z priečinka `configs/`. Vybraný konfig určuje:

- experimentálnu konfiguráciu použitú pri trénovaní modelov;
- cesty k uloženým natrénovaným modelom;
- dátové archívy použité na finálnu predikciu;
- prahovú hodnotu pre binarizáciu pravdepodobnostnej mapy;
- výstupný priečinok pre výsledky.

Kompletné tréningové dáta nie sú súčasťou repozitára z dôvodu ich veľkosti. Tréningová časť je preto v notebooku ponechaná ako štruktúra pipeline, zatiaľ čo spustiteľná časť začína finálnou predikciou pomocou uložených modelov.

## Štruktúra repozitára

Notebook očakáva nasledujúcu štruktúru repozitára:

```text
repo/
├── configs/
│   ├── CH_standard.json
│   ├── CH_low.json
│   ├── CH_region_growth.json
│   ├── CH_2025_raw.json
│   ├── CH_2025_processed.json
│   ├── AR_standard.json
│   ├── AR_low.json
│   └── AR_spoca.json
│
├── src/
│   ├── model_scss_net.py
│   ├── v2_modified_model_scss_net.py
│   └── metrics.py
│
├── trained_models/
├── data/
├── notebooks/
└── outputs/
```

## 1. Nastavenie prostredia a výber konfigurácie

V premennej `CONFIG_PATH` je potrebné zvoliť konfiguračný súbor experimentu. Pri spustení z priečinka `notebooks/` aj z koreňa repozitára sa používajú relatívne cesty voči koreňu repozitára.

In [ ]:
from pathlib import Path
import sys
import json
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# =========================
# VÝBER KONFIGURÁCIE
# =========================
CONFIG_PATH = "configs/CH_standard.json"

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
CONFIG_FILE = PROJECT_ROOT / CONFIG_PATH

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Konfiguračný súbor neexistuje: {CONFIG_FILE}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

TRAINING_CFG = CONFIG["training_config"]
PREDICTION_CFG = CONFIG["prediction_config"]

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

print("Načítaná konfigurácia:", CONFIG_FILE)
print("Experiment:", CONFIG["experiment_id"])
print("Štruktúra:", CONFIG["structure"])
print("Popis:", CONFIG.get("description", ""))

## 2. Import modelov, metrík a základných parametrov

V tejto časti sa načítajú implementácie modelov, metrík a hodnoty z konfiguračného súboru. Rovnaký notebook tak môže spustiť viac experimentálnych variantov bez úpravy samotného pipeline.

In [ ]:
from model_scss_net import scss_net
from v2_modified_model_scss_net import scss_net_convlstm_early
from metrics import dice_soft, iou_soft, dice_np, iou_np, bce_dice_loss

LOSS_FN = bce_dice_loss

PHENOMENON = CONFIG["structure"]
CHANNEL = CONFIG["channel"]

MODEL_PARAMS = TRAINING_CFG["model_parameters"]

IMG_SIZE = int(TRAINING_CFG["image_size"][0])
T_STEPS = int(TRAINING_CFG["sequence_length"])
HIST_T = int(TRAINING_CFG["previous_frames"])
CHANNELS = 1
USE_CURRENT_FRAME = bool(TRAINING_CFG.get("target_frame_included", True))

FILTERS = int(MODEL_PARAMS["filters"])
LAYERS = int(MODEL_PARAMS["encoder_decoder_levels"])
DROP_PROB = float(MODEL_PARAMS["dropout"])
CONVLSTM_FILTERS = int(MODEL_PARAMS["convlstm_filters"])
BATCH_SIZE_BASELINE = int(MODEL_PARAMS.get("batch_size_baseline", 8))
BATCH_SIZE_CONVLSTM = int(MODEL_PARAMS.get("batch_size_convlstm", 4))

THRESHOLD = float(PREDICTION_CFG["threshold"])
OUT_DIR = PROJECT_ROOT / PREDICTION_CFG["output_dir"]
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_FULL_TRAINING = False
RUN_FULL_TEST_EVALUATION = False

print("Image size:", IMG_SIZE)
print("Sequence length:", T_STEPS)
print("Filters:", FILTERS)
print("Layers:", LAYERS)
print("ConvLSTM filters:", CONVLSTM_FILTERS)
print("Threshold:", THRESHOLD)
print("Output dir:", OUT_DIR)

## 3. Dokumentácia tréningovej konfigurácie

Táto časť neštartuje trénovanie. Slúži na zobrazenie toho, v akých podmienkach boli uložené modely natrénované. Kompletná tréningová výberka vrátane časových sekvencií nie je v repozitári zahrnutá.

In [ ]:
print("Tréningové dáta v repozitári:", TRAINING_CFG["training_data_in_repository"])
print("Tréning zapnutý v repozitári:", TRAINING_CFG["training_enabled_in_repository"])
print("Tréningová výberka:", TRAINING_CFG["train_dataset"])
print("Zdroje anotácií:", ", ".join(TRAINING_CFG["annotation_sources"]))
print("Kanál:", TRAINING_CFG["image_channel"])
print("Časové odstupy:", TRAINING_CFG["time_offsets_minutes"])
print("Predspracovanie:", TRAINING_CFG["preprocessing"])
print("Poznámka:", TRAINING_CFG.get("note", ""))

## 4. Zástupná časť pre kompletné tréningové dáta

V pôvodných experimentoch tu prebiehalo vyhľadanie tréningových obrazov, masiek a časových sekvencií. V tomto repozitári sú tieto priečinky ponechané iba ako zástupné cesty, pretože kompletná rozšírená tréningová výberka má veľký objem.

In [ ]:
FULL_DATA_ROOT = PROJECT_ROOT / "full_experiment_data_not_included"

if PHENOMENON == "CH":
    TRAIN_DIR = FULL_DATA_ROOT / "193_temporal_T3" / "193_train"
    TEST_DIR = FULL_DATA_ROOT / "193_temporal_T3" / "193_test"
    FRAMES_DIR = FULL_DATA_ROOT / "193_temporal_T3" / "ch_frames"
else:
    TRAIN_DIR = FULL_DATA_ROOT / "171_temporal_T3" / "171_train"
    TEST_DIR = FULL_DATA_ROOT / "171_temporal_T3" / "171_test"
    FRAMES_DIR = FULL_DATA_ROOT / "171_temporal_T3" / "ar_frames"

print("Zástupná tréningová cesta:", TRAIN_DIR)
print("Zástupná testovacia cesta:", TEST_DIR)
print("Zástupná cesta sekvencií:", FRAMES_DIR)

## 5. Vytvorenie referenčných modelov podľa konfigurácie

Modely sa v tejto časti iba zostavia podľa parametrov z konfigurácie. Skutočné váhy pre finálnu predikciu sa neskôr načítajú zo súborov `.keras`.

In [ ]:
def build_baseline_model():
    try:
        model = scss_net(
            input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
            filters=FILTERS,
            layers=LAYERS,
            drop_prob=DROP_PROB,
        )
    except TypeError:
        model = scss_net(input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=LOSS_FN,
        metrics=[dice_soft, iou_soft],
    )
    return model


def build_convlstm_model():
    try:
        model = scss_net_convlstm_early(
            input_shape=(T_STEPS, IMG_SIZE, IMG_SIZE, CHANNELS),
            filters=FILTERS,
            layers=LAYERS,
            drop_prob=DROP_PROB,
            convlstm_filters=CONVLSTM_FILTERS,
        )
    except TypeError:
        model = scss_net_convlstm_early(
            input_shape=(T_STEPS, IMG_SIZE, IMG_SIZE, CHANNELS)
        )

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=LOSS_FN,
        metrics=[dice_soft, iou_soft],
    )
    return model


baseline_reference = build_baseline_model()
convlstm_reference = build_convlstm_model()

print("Referenčný základný model:", baseline_reference.input_shape, "->", baseline_reference.output_shape)
print("Referenčný ConvLSTM model:", convlstm_reference.input_shape, "->", convlstm_reference.output_shape)

## 6. Zástupná časť pre trénovanie

Ak by bola kompletná tréningová výberka prítomná, v tejto časti by prebiehalo trénovanie základného modelu aj modelu s časovým kontextom. V demonštračnej verzii sa táto časť preskočí.

In [ ]:
if RUN_FULL_TRAINING:
    raise NotImplementedError(
        "Kompletné trénovanie vyžaduje plnú časovú tréningovú výberku, "
        "ktorá nie je súčasťou repozitára."
    )
else:
    print("Trénovanie je v tomto repozitári preskočené.")
    print("Na finálnu predikciu sa použijú uložené natrénované modely z konfigurácie.")

## 7. Rozbalenie dát pre finálnu predikciu

Konfiguračný súbor obsahuje zoznam archívov potrebných pre daný experiment. Notebook ich rozbalí do výstupného priečinka konkrétneho experimentu.

In [ ]:
def unzip_archives(archive_paths, extract_root: Path):
    extract_root.mkdir(parents=True, exist_ok=True)

    for rel_path in archive_paths:
        zip_path = PROJECT_ROOT / rel_path
        if not zip_path.exists():
            raise FileNotFoundError(f"Chýba dátový archív: {zip_path}")

        archive_extract_dir = extract_root / zip_path.stem
        archive_extract_dir.mkdir(parents=True, exist_ok=True)

        existing = [p for p in archive_extract_dir.rglob("*") if p.is_file()]
        if existing:
            print(f"Už rozbalené: {archive_extract_dir} ({len(existing)} súborov)")
            continue

        print(f"Rozbaľujem {zip_path.name} -> {archive_extract_dir}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(archive_extract_dir)

    return extract_root


EXTRACT_DIR = OUT_DIR / "extracted_prediction_data"
unzip_archives(PREDICTION_CFG["archives"], EXTRACT_DIR)

print("Dáta rozbalené do:", EXTRACT_DIR)

## 8. Vyhľadanie obrazov, masiek a časových sekvencií

Keďže dátové archívy môžu byť uložené rôznym spôsobom, notebook po rozbalení automaticky vyhľadá obrazové súbory, masky a priečinky so sekvenciami `input_1`, `input_2`, `input_3`.

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def list_image_files(folder: Path):
    return sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ])

def path_has_keyword(path: Path, keywords):
    parts = [part.lower() for part in path.parts]
    stem = path.stem.lower()
    return any(k in stem for k in keywords) or any(any(k in part for k in keywords) for part in parts)

def find_sequence_folders(root: Path):
    return sorted([
        p for p in root.rglob("*")
        if p.is_dir() and all((p / f"input_{i}.png").exists() for i in range(1, HIST_T + 1))
    ])

def is_inside_any(path: Path, folders):
    try:
        return any(folder in path.parents for folder in folders)
    except Exception:
        return False

all_files = list_image_files(EXTRACT_DIR)
sequence_folders = find_sequence_folders(EXTRACT_DIR)

mask_files = [p for p in all_files if path_has_keyword(p, ["mask", "masks"])]
sequence_files = [p for p in all_files if is_inside_any(p, sequence_folders)]

image_files = [
    p for p in all_files
    if p not in mask_files
    and p not in sequence_files
    and not path_has_keyword(p, ["mask", "masks", "sequence", "sequences", "seq"])
]

print("Počet nájdených obrazov:", len(image_files))
print("Počet nájdených masiek:", len(mask_files))
print("Počet nájdených sekvencií:", len(sequence_folders))

if len(image_files) == 0 or len(mask_files) == 0:
    raise RuntimeError(
        "Nepodarilo sa automaticky nájsť obrazy alebo masky. "
        "Skontrolujte vnútornú štruktúru dátového archívu."
    )

In [ ]:
def normalize_key(path: Path):
    stem = path.stem
    replacements = [
        "_mask", "_masks", "_image", "_target",
        "mask_", "image_", "target_",
        "baseline_", "convlstm_"
    ]
    for r in replacements:
        stem = stem.replace(r, "")
    return stem.lower()

image_map = {normalize_key(p): p for p in image_files}
mask_map = {normalize_key(p): p for p in mask_files}
sequence_map = {normalize_key(p): p for p in sequence_folders}

common_keys = sorted(set(image_map) & set(mask_map))

rows = []
if common_keys:
    for idx, key in enumerate(common_keys):
        seq = sequence_map.get(key)
        if seq is None and idx < len(sequence_folders):
            seq = sequence_folders[idx]
        rows.append({
            "sample_id": key,
            "image_path": image_map[key],
            "mask_path": mask_map[key],
            "sequence_dir": seq
        })
else:
    n = min(len(image_files), len(mask_files))
    for idx in range(n):
        seq = sequence_folders[idx] if idx < len(sequence_folders) else None
        rows.append({
            "sample_id": f"sample_{idx:04d}",
            "image_path": image_files[idx],
            "mask_path": mask_files[idx],
            "sequence_dir": seq
        })

final_df = pd.DataFrame(rows)

print("Počet spárovaných vzoriek:", len(final_df))
display(final_df.head())

## 9. Načítanie uložených modelov

Cesty k modelom sú definované v časti `prediction_config` vybraného konfiguračného súboru.

In [ ]:
BASELINE_MODEL_PATH = PROJECT_ROOT / PREDICTION_CFG["models"]["baseline"]
CONVLSTM_MODEL_PATH = PROJECT_ROOT / PREDICTION_CFG["models"]["convlstm"]

for model_path in [BASELINE_MODEL_PATH, CONVLSTM_MODEL_PATH]:
    if not model_path.exists():
        raise FileNotFoundError(f"Chýba model: {model_path}")

custom_objects = {
    "dice_soft": dice_soft,
    "iou_soft": iou_soft,
    "bce_dice_loss": bce_dice_loss,
}

baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH, custom_objects=custom_objects)
convlstm_model = tf.keras.models.load_model(CONVLSTM_MODEL_PATH, custom_objects=custom_objects)

print("Načítaný základný model:", BASELINE_MODEL_PATH)
print("Načítaný ConvLSTM model:", CONVLSTM_MODEL_PATH)

## 10. Príprava vstupov pre finálnu predikciu

Základný model dostáva jeden cieľový obraz. Model ConvLSTM-SCSS-Net dostáva sekvenciu troch predchádzajúcich snímok a cieľového obrazu.

In [ ]:
def load_gray_float(path: Path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr[..., None]

def load_mask_float(path: Path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.NEAREST)
    arr = (np.asarray(img, dtype=np.float32) > 127).astype(np.float32)
    return arr[..., None]

def load_sequence(sequence_folder: Path, target_image_path: Path):
    frames = []
    for i in range(1, HIST_T + 1):
        frame_path = sequence_folder / f"input_{i}.png"
        if frame_path.exists():
            frames.append(load_gray_float(frame_path))
        else:
            frames.append(load_gray_float(target_image_path))

    if USE_CURRENT_FRAME:
        frames.append(load_gray_float(target_image_path))

    while len(frames) < T_STEPS:
        frames.append(load_gray_float(target_image_path))

    return np.stack(frames[:T_STEPS], axis=0)


X_base_final = []
X_temp_final = []
Y_final = []
sample_ids = []

for _, row in final_df.iterrows():
    target = load_gray_float(row["image_path"])
    mask = load_mask_float(row["mask_path"])

    if row["sequence_dir"] is not None:
        seq = load_sequence(row["sequence_dir"], row["image_path"])
    else:
        seq = np.stack([target] * T_STEPS, axis=0)

    X_base_final.append(target)
    X_temp_final.append(seq)
    Y_final.append(mask)
    sample_ids.append(row["sample_id"])

X_base_final = np.asarray(X_base_final, dtype=np.float32)
X_temp_final = np.asarray(X_temp_final, dtype=np.float32)
Y_final = np.asarray(Y_final, dtype=np.float32)

print("X_base_final:", X_base_final.shape)
print("X_temp_final:", X_temp_final.shape)
print("Y_final:", Y_final.shape)

## 11. Finálna predikcia

Modely vytvoria pravdepodobnostné mapy. Tie sa následne prevedú na binárne masky pomocou prahovej hodnoty definovanej v konfigurácii.

In [ ]:
base_prob_final = baseline_model.predict(X_base_final, verbose=1)
temp_prob_final = convlstm_model.predict(X_temp_final, verbose=1)

base_pred_final = (base_prob_final > THRESHOLD).astype(np.float32)
temp_pred_final = (temp_prob_final > THRESHOLD).astype(np.float32)

print("Použitý threshold:", THRESHOLD)
print("Baseline predictions:", base_pred_final.shape)
print("ConvLSTM predictions:", temp_pred_final.shape)

## 12. Numerické porovnanie

Okrem metrík Dice a IoU voči dostupnej maske sa ukladajú aj hodnoty porovnávajúce modely priamo medzi sebou, napríklad rozdiel predikovanej plochy a IoU medzi predikciami modelov.

In [ ]:
def model_to_model_iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float((inter + 1e-7) / (union + 1e-7))

rows = []
for i, sample_id in enumerate(sample_ids):
    y = Y_final[i, ..., 0]
    bp = base_pred_final[i, ..., 0]
    tp = temp_pred_final[i, ..., 0]

    rows.append({
        "sample_id": sample_id,
        "baseline_dice": dice_np(y, bp),
        "baseline_iou": iou_np(y, bp),
        "convlstm_dice": dice_np(y, tp),
        "convlstm_iou": iou_np(y, tp),
        "baseline_area_ratio": float(bp.mean()),
        "convlstm_area_ratio": float(tp.mean()),
        "delta_area_convlstm_minus_baseline": float(tp.mean() - bp.mean()),
        "iou_between_models": model_to_model_iou(bp, tp),
        "xor_ratio": float(np.logical_xor(bp.astype(bool), tp.astype(bool)).mean()),
    })

metrics_df = pd.DataFrame(rows)
metrics_path = OUT_DIR / f"{CONFIG['experiment_id']}_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df.head())
display(metrics_df.describe())
print("Metriky uložené do:", metrics_path)

## 13. Vizuálne porovnanie

Overlay vizualizácia používa nasledujúce farby:

- červená: pixely označené iba základným modelom SCSS-Net;
- modrá: pixely označené iba modelom ConvLSTM-SCSS-Net;
- žltá: pixely označené oboma modelmi.

In [ ]:
def difference_rgb(base_mask, conv_mask):
    base = np.squeeze(base_mask).astype(bool)
    conv = np.squeeze(conv_mask).astype(bool)

    rgb = np.zeros((base.shape[0], base.shape[1], 3), dtype=np.uint8)
    rgb[base & ~conv] = [255, 0, 0]
    rgb[conv & ~base] = [0, 180, 255]
    rgb[base & conv] = [255, 255, 0]
    return rgb

def plot_final_comparison(i):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[0].set_title("Obraz")

    axes[1].imshow(Y_final[i, ..., 0], cmap="gray")
    axes[1].set_title("Maska")

    axes[2].imshow(base_pred_final[i, ..., 0], cmap="gray")
    axes[2].set_title("SCSS-Net")

    axes[3].imshow(temp_pred_final[i, ..., 0], cmap="gray")
    axes[3].set_title("ConvLSTM-SCSS-Net")

    axes[4].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[4].imshow(difference_rgb(base_pred_final[i], temp_pred_final[i]), alpha=0.55)
    axes[4].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(str(sample_ids[i]))
    plt.tight_layout()
    plt.show()

for i in range(min(5, len(sample_ids))):
    plot_final_comparison(i)

## 14. Uloženie výstupov

Notebook uloží binárne masky oboch modelov, vizuálne porovnania a kópiu použitej konfigurácie do výstupného priečinka experimentu.

In [ ]:
VIS_DIR = OUT_DIR / "visual_comparisons"
BASE_MASK_DIR = OUT_DIR / "baseline_masks"
TEMP_MASK_DIR = OUT_DIR / "convlstm_masks"

VIS_DIR.mkdir(parents=True, exist_ok=True)
BASE_MASK_DIR.mkdir(parents=True, exist_ok=True)
TEMP_MASK_DIR.mkdir(parents=True, exist_ok=True)

for i, sample_id in enumerate(sample_ids):
    base_img = Image.fromarray((base_pred_final[i, ..., 0] * 255).astype(np.uint8))
    temp_img = Image.fromarray((temp_pred_final[i, ..., 0] * 255).astype(np.uint8))

    base_img.save(BASE_MASK_DIR / f"{sample_id}_baseline_pred.png")
    temp_img.save(TEMP_MASK_DIR / f"{sample_id}_convlstm_pred.png")

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(X_base_final[i, ..., 0], cmap="gray"); axes[0].set_title("Obraz")
    axes[1].imshow(Y_final[i, ..., 0], cmap="gray"); axes[1].set_title("Maska")
    axes[2].imshow(base_pred_final[i, ..., 0], cmap="gray"); axes[2].set_title("SCSS-Net")
    axes[3].imshow(temp_pred_final[i, ..., 0], cmap="gray"); axes[3].set_title("ConvLSTM-SCSS-Net")
    axes[4].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[4].imshow(difference_rgb(base_pred_final[i], temp_pred_final[i]), alpha=0.55)
    axes[4].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(str(sample_id))
    plt.tight_layout()
    plt.savefig(VIS_DIR / f"{sample_id}_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

with open(OUT_DIR / "used_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("Výstupy uložené do:", OUT_DIR)
print("Masky základného modelu:", BASE_MASK_DIR)
print("Masky ConvLSTM modelu:", TEMP_MASK_DIR)
print("Vizuálne porovnania:", VIS_DIR)

## 15. Zhrnutie

Notebook bol spustený s konfiguráciou uvedenou v premennej `CONFIG_PATH`. Ak chcete spustiť iný experiment, zmeňte túto hodnotu na iný JSON súbor v priečinku `configs/` a spustite notebook znova.